In [0]:
spark.sql("CREATE TABLE IF NOT EXISTS mano_catalog.gold.sample_table_SCD1 (id INT, name STRING, value DOUBLE, gender STRING, country STRING, update_timestamp TIMESTAMP) USING DELTA PARTITIONED BY (country, gender)")

data = [
    (1, "Alice", 12.4, "Female", "USA"),
    (2, "Bob", 24.7, "Male", "UK"),
    (3, "Charlie", 36.1, "Male", "India"),
    (4, "Diana", 48.5, "Female", "USA"),
    (5, "Eve", 59.8, "Female", "UK")
]

from pyspark.sql.functions import current_timestamp

df = spark.createDataFrame(data, ["id", "name", "value", "gender", "country"])
df = df.withColumn("update_timestamp", current_timestamp())
df.write.mode("append").saveAsTable("mano_catalog.gold.sample_table_SCD1")
display(spark.table("mano_catalog.gold.sample_table_SCD1"))


In [0]:
# SCD Type 1 MERGE for mano_catalog.gold.sample_table_SCD1
# SCD1 overwrites existing records with the latest values (no history preserved)

# Step 1: Create sample incoming/staging data (new + updated records)
new_data = [
    (1, "Alice", 15.0, "Female", "USA"),    # Updated value for existing id=1
    (3, "Charlie", 40.0, "Male", "India"),   # Updated value for existing id=3
    (6, "Frank", 72.3, "Male", "USA"),       # New record
    (7, "Grace", 88.1, "Female", "India"),   # New record
]

from pyspark.sql.functions import current_timestamp

df_updates = spark.createDataFrame(new_data, ["id", "name", "value", "gender", "country"])
df_updates = df_updates.withColumn("update_timestamp", current_timestamp())

# Step 2: Write staging data to a temp view
df_updates.createOrReplaceTempView("staging_table")

# Step 3: Execute SCD Type 1 MERGE
spark.sql("""
    MERGE INTO mano_catalog.gold.sample_table_SCD1 AS target
    USING staging_table AS source
    ON target.id = source.id
    WHEN MATCHED AND (
        target.name <> source.name OR
        target.value <> source.value OR
        target.gender <> source.gender OR
        target.country <> source.country
    ) THEN
        UPDATE SET
            target.name = source.name,
            target.value = source.value,
            target.gender = source.gender,
            target.country = source.country,
            target.update_timestamp = source.update_timestamp
    WHEN NOT MATCHED THEN
        INSERT (id, name, value, gender, country, update_timestamp)
        VALUES (source.id, source.name, source.value, source.gender, source.country, source.update_timestamp)
""")



In [0]:
# Step 4: Display the results after merge
display(spark.table("mano_catalog.gold.sample_table_SCD1").orderBy("id"))

In [0]:
# -------------------------------------------------------
# SCD Type 2 Implementation for mano_catalog.gold.sample_table_SCD2
# SCD2 preserves full history of changes with audit fields
# -------------------------------------------------------

# Step 1: Create the SCD2 target table with audit columns
spark.sql("""
    CREATE TABLE IF NOT EXISTS mano_catalog.gold.sample_table_SCD2 (
        surrogate_key BIGINT GENERATED ALWAYS AS IDENTITY,
        id INT,
        name STRING,
        value DOUBLE,
        gender STRING,
        country STRING,
        -- SCD2 tracking columns
        effective_start_date TIMESTAMP,
        effective_end_date TIMESTAMP,
        is_current BOOLEAN,
        -- Audit fields
        created_by STRING,
        created_timestamp TIMESTAMP,
        updated_by STRING,
        updated_timestamp TIMESTAMP,
        record_source STRING,
        record_hash STRING
    ) USING DELTA
    PARTITIONED BY (country, gender)
""")

print("✅ SCD2 table created successfully.")

# Step 2: Load initial data into SCD2 table (from the existing SCD1 table)
spark.sql("""
    INSERT INTO mano_catalog.gold.sample_table_SCD2 (
        id, name, value, gender, country,
        effective_start_date, effective_end_date, is_current,
        created_by, created_timestamp, updated_by, updated_timestamp,
        record_source, record_hash
    )
    SELECT
        id, name, value, gender, country,
        current_timestamp() AS effective_start_date,
        CAST('9999-12-31 23:59:59' AS TIMESTAMP) AS effective_end_date,
        TRUE AS is_current,
        current_user() AS created_by,
        current_timestamp() AS created_timestamp,
        current_user() AS updated_by,
        current_timestamp() AS updated_timestamp,
        'initial_load' AS record_source,
        md5(concat_ws('|', CAST(id AS STRING), name, CAST(value AS STRING), gender, country)) AS record_hash
    FROM mano_catalog.gold.sample_table_SCD1
""")

print("✅ Initial data loaded into SCD2 table.")
display(spark.table("mano_catalog.gold.sample_table_SCD2").orderBy("id", "effective_start_date"))

# Step 3: SCD2 MERGE with new/updated records
new_data = [
    (1, "Alice", 20.0, "Female", "USA"),     # Updated value for existing id=1
    (3, "Charlie", 50.0, "Male", "India"),    # Updated value for existing id=3
    (8, "Henry", 95.5, "Male", "UK"),         # Brand new record
    (9, "Irene", 110.2, "Female", "India"),   # Brand new record
]

from pyspark.sql.functions import current_timestamp, md5, concat_ws, col, lit

df_scd2_updates = spark.createDataFrame(new_data, ["id", "name", "value", "gender", "country"])
df_scd2_updates = df_scd2_updates.withColumn("record_hash",
    md5(concat_ws("|", col("id").cast("string"), col("name"), col("value").cast("string"), col("gender"), col("country"))))
df_scd2_updates.createOrReplaceTempView("scd2_staging")

# Step 4: Execute SCD Type 2 MERGE
# Phase A — Expire (close out) existing current records that have changes
spark.sql("""
    MERGE INTO mano_catalog.gold.sample_table_SCD2 AS target
    USING (
        SELECT s.*
        FROM scd2_staging s
        INNER JOIN mano_catalog.gold.sample_table_SCD2 t
            ON s.id = t.id
            AND t.is_current = TRUE
            AND s.record_hash <> t.record_hash
    ) AS source
    ON target.id = source.id AND target.is_current = TRUE
    WHEN MATCHED THEN
        UPDATE SET
            target.effective_end_date = current_timestamp(),
            target.is_current = FALSE,
            target.updated_by = current_user(),
            target.updated_timestamp = current_timestamp()
""")

print("✅ Phase A complete: Expired changed records.")

# Phase B — Insert new current versions for changed records + brand new records
spark.sql("""
    INSERT INTO mano_catalog.gold.sample_table_SCD2 (
        id, name, value, gender, country,
        effective_start_date, effective_end_date, is_current,
        created_by, created_timestamp, updated_by, updated_timestamp,
        record_source, record_hash
    )
    SELECT
        s.id, s.name, s.value, s.gender, s.country,
        current_timestamp() AS effective_start_date,
        CAST('9999-12-31 23:59:59' AS TIMESTAMP) AS effective_end_date,
        TRUE AS is_current,
        current_user() AS created_by,
        current_timestamp() AS created_timestamp,
        current_user() AS updated_by,
        current_timestamp() AS updated_timestamp,
        CASE
            WHEN t.id IS NULL THEN 'new_record'
            ELSE 'scd2_update'
        END AS record_source,
        s.record_hash
    FROM scd2_staging s
    LEFT JOIN mano_catalog.gold.sample_table_SCD2 t
        ON s.id = t.id AND t.is_current = TRUE
    WHERE t.id IS NULL  -- Brand new records (not in target)
       OR s.record_hash <> t.record_hash  -- This won't match now since we expired in Phase A
       OR NOT EXISTS (
           SELECT 1 FROM mano_catalog.gold.sample_table_SCD2 e
           WHERE e.id = s.id AND e.is_current = TRUE
       )  -- Records that were expired in Phase A
""")

print("✅ Phase B complete: Inserted new current versions.")
print("\n📋 Final SCD2 Table (all historical + current records):")
display(spark.table("mano_catalog.gold.sample_table_SCD2").orderBy("id", "effective_start_date"))